# Objaverse-XL subset download
Use the parameters below to pick how many objects to pull and where to store them under `data/`.


In [5]:
import objaverse.xl as oxl
import pandas as pd

ANNOTATION_CACHE = "data/objaverse_annotations"
OUTPUT_DIR = "data/my_objaverse_subset"  # change if you want a different folder
PER_SOURCE = 2  # objects per source; set to None to download everything (very large)
RANDOM_SEED = 0

# Load cached annotations (downloaded once and reused).
annotations = oxl.get_annotations(download_dir=ANNOTATION_CACHE)
print(f"Loaded {len(annotations):,} annotations")

# Pick a subset to download.
if PER_SOURCE:
    objects_df = (
        annotations
        .groupby("source", group_keys=False)
        .apply(lambda df: df.sample(n=min(PER_SOURCE, len(df)), random_state=RANDOM_SEED))
        .reset_index(drop=True)
    )
else:
    objects_df = annotations.copy()

print(f"Prepared {len(objects_df):,} objects to download")
objects_df.head()



Loaded 9,767,011 annotations
Prepared 8 objects to download


/var/folders/q5/bq_syrbd6dqdpff08pr4cv9m0000gn/T/ipykernel_4789/8706720.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: df.sample(n=min(PER_SOURCE, len(df)), random_state=RANDOM_SEED))


,fileIdentifier,source,license,fileType,sha256,metadata
0,https://github.com/ThomasRouvinez/Hydraulics-A...,github,None,fbx,29b965d97325b46419ae6021afa2f7bf39c4581deb2def...,{}
1,https://github.com/Macix97/paradise-city/blob/...,github,GNU General Public License v3.0,fbx,207b51760177cd30497ead0a31e19fcb66e767baeb41e4...,{}
2,https://sketchfab.com/3d-models/e8119fbb940c45...,sketchfab,Creative Commons - Attribution,glb,6ecc1b3f3f51dfc8e7c594114344dc7a6ece33d2413fd3...,{}
3,https://sketchfab.com/3d-models/75a1f71afb3b46...,sketchfab,Creative Commons - Attribution,glb,e7561104809ce1852cba5111e0662b09468855cef1dd6f...,{}
4,https://3d-api.si.edu/content/document/3d_pack...,smithsonian,Creative Commons Zero v1.0 Universal,glb,62a4dfb7321ca9043d9383f3be61916b155ab7e337ce09...,"{""title"": ""Eulemur albifrons: Mandible""}"


## Checking available categories

In [2]:
import objaverse
import re

lvis = objaverse.load_lvis_annotations() #! In LVIS not OXL
objaverseOxl = oxl.get_annotations

labels = sorted(lvis.keys())

print(f"Total labels: {len(labels)}")
for name in labels[900:1156]:
    print(f"- {name}")

Total labels: 1156
- shirt
- shoe
- shopping_bag
- shopping_cart
- short_pants
- shot_glass
- shoulder_bag
- shovel
- shower_cap
- shower_curtain
- shower_head
- shredder_(for_paper)
- signboard
- silo
- sink
- skateboard
- skewer
- ski
- ski_boot
- ski_parka
- ski_pole
- skirt
- skullcap
- sled
- sleeping_bag
- slide
- slipper_(footwear)
- smoothie
- snake
- snowboard
- snowman
- snowmobile
- soap
- soccer_ball
- sock
- sofa
- sofa_bed
- softball
- solar_array
- sombrero
- soup
- soup_bowl
- soupspoon
- soya_milk
- space_shuttle
- sparkler_(fireworks)
- spatula
- speaker_(stero_equipment)
- spear
- spectacles
- spice_rack
- spider
- sponge
- spoon
- sportswear
- spotlight
- squid_(food)
- squirrel
- stagecoach
- stapler_(stapling_machine)
- starfish
- statue_(sculpture)
- steak_(food)
- steak_knife
- steering_wheel
- step_stool
- stepladder
- stereo_(sound_system)
- stew
- stirrer
- stirrup
- stool
- stop_sign
- stove
- strainer
- strap
- straw_(for_drinking)
- strawberry
- street_sig

## Downloading

So when I actually try to doanload off of OXL version it get more random objects probably because its large, and there were some errors tryna get them so i mixed LVIS + OXL together, getting the UID's from LVIS and then downlaoding off of OXL.

In [3]:
import objaverse
import objaverse.xl as oxl
import re

# ANNOTATION_CACHE = "data/objaverse_annotations"
OUTPUT_DIR = "data/objaverse_found"
N = 10

# Get LVIS annotations
lvis = objaverse.load_lvis_annotations()
terms = [
"chair",
"sofa",
"loveseat",
"armchair",
"bench",
"stool",
"couch", # not in LVIS, but keep if you also search metadata
"table",
"coffee_table",
"dining_table",
"kitchen_table",
"desk",
"cabinet",
"cupboard",
"dresser",
"wardrobe",
"armoire",
"bookcase",
"shelf",
"bed",
"bunk_bed",
"sofa_bed",
"nightstand", # not in LVIS; add if you search metadata
"lamp",
"table_lamp",
"chandelier",
"lantern",
"mirror",
"clock",
"wall_clock",
"curtain",
"pillow",
"blanket",
"bedspread",
"towel",
"rug", # not in LVIS; add if searching metadata
"picture", # not in LVIS; add if searching metadata
"painting"
]

banned_terms = [
"streetlight",
"street_sign",
"traffic_light",
"lamppost",
"stop_sign",
"parking_meter",
"bus_(vehicle)",
"car_(automobile)",
"truck",
"train_(railroad_vehicle)",
"motorcycle",
"bicycle",
"boat"
]

def has_token(name: str) -> bool:
    tokens = re.split(r"[^a-z]+", name.lower())
    return any(tok in terms for tok in tokens)

categories = [k for k in lvis.keys() if has_token(k)]
print(f"related categories: {categories}")

uids = set()
for cat in categories:
    uids.update(lvis.get(cat, []))
print(f"Total UIDs from LVIS: {len(uids)}")
print(f"Sample UIDs: {list(uids)[:3]}")

# Load cached annotations
# annotations = oxl.get_annotations(download_dir=ANNOTATION_CACHE)
sketchfab_df = annotations[annotations["source"] == "sketchfab"].copy()

# Extract UID from fileIdentifier URL
# URL format: https://sketchfab.com/3d-models/3fc08fd6544d4add8ce55f6c5e2bc872
sketchfab_df["uid"] = sketchfab_df["fileIdentifier"].str.split("/").str[-1]

print(f"Sample extracted UIDs: {sketchfab_df['uid'].head(3).tolist()}")

# Now filter by matching UIDs
df = sketchfab_df[sketchfab_df["uid"].isin(uids)]
print(f"Matched: {len(df)}")

# Limit and download
if len(df) > 0:
    df_limited = df.sample(n=min(N, len(df)), random_state=42)
    print(f"Downloading {len(df_limited)} trees")
    
    oxl.download_objects(
        objects=df_limited,
        download_dir=OUTPUT_DIR,
        processes=8
    )
else:
    print("No matches found")

related categories: ['alarm_clock', 'armchair', 'armoire', 'bath_towel', 'bed', 'bedspread', 'bench', 'blanket', 'bookcase', 'bunk_bed', 'cabinet', 'chair', 'chandelier', 'clock', 'clock_tower', 'coffee_table', 'cupboard', 'curtain', 'deck_chair', 'desk', 'dining_table', 'dresser', 'file_cabinet', 'folding_chair', 'hand_towel', 'kitchen_table', 'lamp', 'lantern', 'loveseat', 'mirror', 'music_stool', 'oil_lamp', 'painting', 'paper_towel', 'pew_(church_bench)', 'pillow', 'pool_table', 'rearview_mirror', 'rocking_chair', 'saddle_blanket', 'shower_curtain', 'sofa', 'sofa_bed', 'step_stool', 'stool', 'table', 'table-tennis_table', 'table_lamp', 'towel', 'towel_rack', 'wall_clock', 'wardrobe']
Total UIDs from LVIS: 3176
Sample UIDs: ['8b491093657e4281865beb3c3fd6fbbf', '5d9db62931a947d48d3c26625b69cbf0', '3b4ee19c627e4fb4a3305621cf925aa2']
Sample extracted UIDs: ['3fc08fd6544d4add8ce55f6c5e2bc872', '2bb529bd1f1f42cf9cdfa8aff56b35f6', 'd09d556a242041a9b65a54255dd57f9b']
Matched: 3141


2025-12-12 13:52:54.499 | INFO     | objaverse.xl.sketchfab:download_objects:508 - Found 10 objects already downloaded
2025-12-12 13:52:54.499 | INFO     | objaverse.xl.sketchfab:download_objects:529 - Downloading 0 new objects across 8 processes


Best ooption for filetering is get tiny lama to loook at the label list (fine tune it on this) and further tune it to select the terms fom a propmt and banned terms to create the AR scene

# Tunning tiny lama to pass the prompts
PineConeDB will store a the below metadata for each label in objaverse:

```
{
  "id": "pine-tree",
  "values": [ ...embedding... ],
  "metadata": {
    "label": "pine-tree",
    "description": "...",
    "category": "vegetation"
  }
}
```
Here the label is the objaverse object label and description is the <40 word description of the label create by a larger LLm. Category is also decided by a larger LLM and added for each 1156 labels. 

In [4]:
# Correcting JSON file structure to be upserted to pinecode helper code
import json

with open("./data/labels.json", "r") as f:
    data = json.load(f)

new_data = []
for i, item in enumerate(data, 1):
    item_cpy = dict(item)
    # Check if keys exist before popping
    if "label" in item_cpy:
        item_cpy["category"] = item_cpy.pop("label")
    if "description" in item_cpy:
        item_cpy["chunk_text"] = item_cpy.pop("description")
    item_cpy["_id"] = f"rec{i}"
    new_data.append(item_cpy)

with open("./data/labels.json", "w") as f:
    # override the file
    json.dump(new_data, f, indent=2)

## Upserting to PineCone DB

In [5]:
from pinecone import (
    Pinecone,
    ServerlessSpec,
    CloudProvider,
    AwsRegion,
    IndexEmbed,
    EmbedModel,
)
from dotenv import load_dotenv
import os


load_dotenv()
PINECONE_API_KEY= os.getenv("PINECONE_API_KEY")
pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "objaverse-index"
if not pc.has_index(index_name):
    pc.create_index_for_model(
        name=index_name,
        cloud=CloudProvider.AWS,
        region=AwsRegion.US_EAST_1,
        embed=IndexEmbed(
            model="llama-text-embed-v2",
            field_map={"text": "chunk_text"},
            metric="cosine",
        ),
    )

# Target the index
dense_index = pc.Index(index_name)

with open("./data/labels.json", "r") as f:
    data = json.load(f)
    
# Upsert the records into a namespace

record = []
for item in data:
    record.append(item)

MAX_BATCH_SIZE = 96
for i in range(0, len(record), MAX_BATCH_SIZE):
    batch = record[i: i + MAX_BATCH_SIZE]
    dense_index.upsert_records("objaverse-namespace", batch)

# Searching the DB with a LLM
So to my understanding, `TinyLlama-1.1B` sits on top of the DB to interpret the results returned. But as you can see from below, the search results are nmo

In [34]:
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)

# To get the unique host for an index, 
# see https://docs.pinecone.io/guides/manage-data/target-an-index
index = pc.Index(name="objaverse-index")

def search_categories(query_text: str, top_k: int = 5):
    results = index.search(
        namespace="objaverse-namespace", 
        query={
            "inputs": {"text": query_text}, 
            "top_k": top_k
        }, # type: ignore
        fields=["category", "chunk_text"]
    )
    return results    

user_prompt = "A lush green tree with a thick trunk and sprawling branches."
search_results = search_categories(user_prompt, top_k=5)
search_results.result["hits"]

[{'_id': 'rec246',
  '_score': 0.38070276379585266,
  'fields': {'category': 'broccoli',
             'chunk_text': 'A green vegetable with a thick stalk and branching '
                           'clusters of small florets that resemble tiny '
                           'trees.'}},
 {'_id': 'rec385',
  '_score': 0.3593895435333252,
  'fields': {'category': 'elephant',
             'chunk_text': 'A very large plant-eating mammal with a long trunk, '
                           'tusks, and large ears.'}},
 {'_id': 'rec578',
  '_score': 0.34882399439811707,
  'fields': {'category': 'koala',
             'chunk_text': 'A small marsupial with round ears, large nose, and '
                           'thick fur, usually shown clinging to a tree trunk.'}},
 {'_id': 'rec50',
  '_score': 0.33739492297172546,
  'fields': {'category': 'bamboo',
             'chunk_text': 'Tall, hollow plant stalks with segmented joints and '
                           'narrow leaves, usually shown as green canes g

# Putting it all together

 we can further refine this using a LLM, the reason is because we need to actually also generate positions for the objects that we have collected so we can place the coherently in the AR world. We also need to download the objects fast so built-in method for downloading is not fast enough research as the entire git repository. So for now putting everything together we will do as above and search in the database and then pass it to an LLM to further refine the list of objects found and also generate their positions vectors. This below script will have to run inside a server which the app can call for a request when I request to send it should respond with a zip folderr containing the GLB file and a positions.TXT file which has the position vector.

In [6]:
# Cacheing Objaverse data and loading cahed annotations
import objaverse.xl as oxl
import pandas as pd

ANNOTATION_CACHE = "data/objaverse_annotations"
OUTPUT_DIR = "data/my_objaverse_subset"  # change if you want a different folder
PER_SOURCE = 2  # objects per source; set to None to download everything (very large)
RANDOM_SEED = 0

annotations = oxl.get_annotations(download_dir=ANNOTATION_CACHE)
print(f"Loaded {len(annotations):,} annotations")

# Pick a subset to download.
if PER_SOURCE:
    objects_df = (
        annotations
        .groupby("source", group_keys=False)
        .apply(lambda df: df.sample(n=min(PER_SOURCE, len(df)), random_state=RANDOM_SEED))
        .reset_index(drop=True)
    )
else:
    objects_df = annotations.copy()

print(f"Prepared {len(objects_df):,} objects to download")
objects_df.head()

Loaded 9,767,011 annotations
Prepared 8 objects to download


/var/folders/q5/bq_syrbd6dqdpff08pr4cv9m0000gn/T/ipykernel_4789/1236012868.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: df.sample(n=min(PER_SOURCE, len(df)), random_state=RANDOM_SEED))


,fileIdentifier,source,license,fileType,sha256,metadata
0,https://github.com/ThomasRouvinez/Hydraulics-A...,github,None,fbx,29b965d97325b46419ae6021afa2f7bf39c4581deb2def...,{}
1,https://github.com/Macix97/paradise-city/blob/...,github,GNU General Public License v3.0,fbx,207b51760177cd30497ead0a31e19fcb66e767baeb41e4...,{}
2,https://sketchfab.com/3d-models/e8119fbb940c45...,sketchfab,Creative Commons - Attribution,glb,6ecc1b3f3f51dfc8e7c594114344dc7a6ece33d2413fd3...,{}
3,https://sketchfab.com/3d-models/75a1f71afb3b46...,sketchfab,Creative Commons - Attribution,glb,e7561104809ce1852cba5111e0662b09468855cef1dd6f...,{}
4,https://3d-api.si.edu/content/document/3d_pack...,smithsonian,Creative Commons Zero v1.0 Universal,glb,62a4dfb7321ca9043d9383f3be61916b155ab7e337ce09...,"{""title"": ""Eulemur albifrons: Mandible""}"


In [ ]:
from zipfile import ZipFile
from pinecone import Pinecone
import os
from urllib.parse import urlparse

PINECONE_API_KEY = os.environ["PINECONE_API_KEY"]
pc = Pinecone(api_key=PINECONE_API_KEY)

# To get the unique host for an index,
# see https://docs.pinecone.io/guides/manage-data/target-an-index
index = pc.Index(name="objaverse-index")


def search_categories(query_text: str, top_k: int = 5):
    results = index.search(
        namespace="objaverse-namespace",
        query={"inputs": {"text": query_text}, "top_k": top_k},  # type: ignore
        fields=["category", "chunk_text"],
    )
    return results


user_prompt = (
    "What I would find in a biology lab that works on teaching about the human body"
)
search_results = search_categories(user_prompt, top_k=5)
hits = search_results.result["hits"]


def format_hits_for_llm(hits):
    lines = []
    for i, h in enumerate(hits):
        cat = h["fields"].get("category", "")
        text = h["fields"].get("chunk_text", "")
        lines.append(f"[{i}] category={cat}\n    chunk_text={text}")
    return "\n\n".join(lines)


def tinyllama_layout(user_prompt: str, hits):
    items_block = format_hits_for_llm(hits)

    prompt = f"""
You are a tool that selects relevant items and assigns 3D positions.

You MUST respond with JSON of exactly this form:

{{
  "layout": {{
    "0": [0.0, 0.05, -0.5],
    "2": [-0.5, 0.05, -0.8]
  }}
}}

Rules:
- "layout" must be a JSON object.
- Each key is a stringified integer index (e.g. "0", "1", "2", ...),
  referring to the indices in square brackets [] in the candidate list below.
- Each value is a JSON array [x, y, z] of floats (meters), suitable for model.position.
- Use only indices that appear in the candidate list.
- Use between 1 and 5 items maximum.
- Keep x and z in [-1.5, 1.5] and y in [0.0, 1.5].
- Do NOT include any other keys.
- Do NOT include explanations or text. Only the JSON object.

User query:
{user_prompt}

Candidates:
{items_block}
"""

    try:
        response = ollama.chat(
            model="tinyllama",
            messages=[
                {
                    "role": "system",
                    "content": (
                        "Always respond with JSON of the form "
                        '{"layout": {"<index>": [x, y, z], ...}} and nothing else.'
                    ),
                },
                {"role": "user", "content": prompt},
            ],
            format="json",
            options={"temperature": 0},
        )
    except:
        raise Exception("Error connecting to Ollama")

    content = response["message"]["content"]
    print("RAW LLM JSON:", content)  # for debugging

    data = json.loads(content)

    if not isinstance(data, dict) or "layout" not in data:
        raise ValueError(f"Unexpected JSON shape from LLM: {data}")

    layout_raw = data["layout"]
    if not isinstance(layout_raw, dict):
        raise ValueError(f"layout is not a dict: {layout_raw}")

    # Convert keys to ints and keep positions as lists
    positions_by_index = {}
    for k, pos in layout_raw.items():
        try:
            idx = int(k)
        except ValueError:
            continue  # skip weird keys

        # Basic sanity checks
        if (
            isinstance(pos, list)
            and len(pos) == 3
            and all(isinstance(v, (int, float)) for v in pos)
            and 0 <= idx < len(hits)
        ):
            positions_by_index[idx] = [float(pos[0]), float(pos[1]), float(pos[2])]

    return positions_by_index


# usage
positions_by_index = tinyllama_layout(user_prompt, hits)
print("Positions by index:", positions_by_index)


import requests

if "name" not in annotations.columns:
    annotations["name"] = (
        annotations["fileIdentifier"]
        .str.extract(r"/([^/]+?)(?:\.[a-zA-Z0-9]+)?$", expand=False)
        .fillna("")
    )

objects_to_display: dict = {}
for key in positions_by_index:
    obj_name = hits[key]["fields"]["category"]
    filtered_objects_data = annotations[
        annotations["name"].str.contains(obj_name, case=False, na=False)
    ]
    objects_to_display[key] = filtered_objects_data


finite_files = []
rows = []

for key, df in objects_to_display.items():
    df = df.iloc[:30].copy()
    df["pos_key"] = key
    rows.append(df)
    for _, row in df.iterrows():
        url = row["fileIdentifier"]

        try:
            head = requests.head(url, allow_redirects=True, timeout=5)
        except requests.RequestException as e:
            print("HEAD failed:", url, e)
            continue

        if head.status_code == 404:
            continue

        # convert GitHub "blob" URL to raw, oxl takes too long
        if (
            "github.com" in url
            and "/blob/" in url
            and "raw.githubusercontent.com" not in url
        ):
            url = url.replace("github.com/", "raw.githubusercontent.com/").replace(
                "/blob/", "/"
            )

        try:
            r = requests.get(url, stream=True, timeout=20)
            r.raise_for_status()
            finite_files.append(url)
        except requests.RequestException as e:
            print("GET failed:", url, e)
            continue

finite_annotations_df = pd.concat(rows, ignore_index=True)
finite_annotations_df = finite_annotations_df.drop_duplicates(subset=["fileIdentifier"])

# finite_annotations_df.iloc[1:5].tail()


def github_blob_to_raw(url: str) -> str:
    if "github.com" in url and "/blob/" in url:
        return url.replace("github.com/", "raw.githubusercontent.com/").replace(
            "/blob/", "/"
        )
    return url


download_dir = "./data/objaverse_found_custom"
os.makedirs(download_dir, exist_ok=True)

bundle_entries = []
local_paths = []

for _, row in finite_annotations_df.iloc[0:10].iterrows():  # type: ignore
    url = github_blob_to_raw(row["fileIdentifier"])
    try:
        r = requests.get(url, stream=True, timeout=20)
        r.raise_for_status()
    except requests.RequestException as e:
        print("GET failed:", url, e)
        continue

    obj_filename = os.path.basename(urlparse(url).path)
    print(f"This is the obj_filename:{obj_filename}")

    out_path_obj = os.path.join(download_dir, obj_filename)
    out_path_txt = os.path.join(download_dir, f"pos_{obj_name}.txt")

    #! --> change to zipping folder and sending with pos.txt
    with open(out_path_obj, "wb") as f:
        for chunk in r.iter_content(8192):
            if chunk:
                f.write(chunk)
    with open(out_path_txt, "w") as f_txt:
        pos = row["pos_key"]
        print(f"This is pos:{pos}")
        if pos is None:
            continue

        f_txt.write(f"{positions_by_index[pos]}")

    bundle_entries.append((out_path_obj, obj_filename))
    bundle_entries.append((out_path_txt, os.path.basename(out_path_txt)))
    zip_path = os.path.join(download_dir, f"assets_bundle_{obj_name}.zip")
    
    with ZipFile(zip_path, "w") as zf:
        for path, arcname in bundle_entries:
            zf.write(path, arcname=arcname)
#     local_paths.append(out_path)

# print("Downloaded:", local_paths)


RAW LLM JSON: {"layout": {
   "0": [0.0, 0.05, -0.5],
   "2": [-0.5, 0.05, -0.8]
}}
Positions by index: {0: [0.0, 0.05, -0.5], 2: [-0.5, 0.05, -0.8]}
This is the obj_filename:Cloth_Hanger_and_Lab_Coat.fbx
This is pos:0
This is the obj_filename:female_lab_coat.fbx
This is pos:0
This is the obj_filename:barbell.fbx
This is pos:2


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/zipfile.py:1550: UserWarning: Duplicate name: 'pos_barbell.txt'
  return self._open_to_write(zinfo, force_zip64=force_zip64)


This is the obj_filename:SM_pancake_for_barbell_01.fbx
This is pos:2
This is the obj_filename:BarbellBench.fbx
This is pos:2
This is the obj_filename:Barbells.fbx
This is pos:2
GET failed: https://raw.githubusercontent.com/ShopyVerse/shopyverse/f3fdd56f549e2df445fcb0d6d5783441e505919c/Assets/Imported Assets/NewAssets/Amazing Mall/Models/Barbell_Ring.fbx 404 Client Error: Not Found for url: https://raw.githubusercontent.com/ShopyVerse/shopyverse/f3fdd56f549e2df445fcb0d6d5783441e505919c/Assets/Imported%20Assets/NewAssets/Amazing%20Mall/Models/Barbell_Ring.fbx
This is the obj_filename:Barbell.gltf.glb
This is pos:2
This is the obj_filename:Barbell.obj
This is pos:2
This is the obj_filename:Barbell.fbx
This is pos:2
Downloaded: ['./data/objaverse_found_custom/Barbell.gltf.glb', './data/objaverse_found_custom/Barbell.gltf.glb', './data/objaverse_found_custom/Barbell.gltf.glb', './data/objaverse_found_custom/Barbell.gltf.glb', './data/objaverse_found_custom/Barbell.gltf.glb', './data/objaver

In [27]:
for _, row in finite_annotations_df.iloc[0:10].iterrows():  # type: ignore
    print(row["pos_key"])
    pos = row["pos_key"]
    print(positions_by_index[pos])


0
[0.0, 0.05, -0.5]
0
[0.0, 0.05, -0.5]
2
[-0.5, 0.05, -0.8]
2
[-0.5, 0.05, -0.8]
2
[-0.5, 0.05, -0.8]
2
[-0.5, 0.05, -0.8]
2
[-0.5, 0.05, -0.8]
2
[-0.5, 0.05, -0.8]
2
[-0.5, 0.05, -0.8]
2
[-0.5, 0.05, -0.8]
